# R3 remaining runs (Kaggle): BM25 baseline, BM25+dense hybrid, multi-seed stability

Runs the experiments deferred in the Stage-3 revision feasibility analysis, mirroring the protocol of
`benchmarks/retrieval/run_r2_benchmark.py` so results are directly comparable to `r2_results.csv` /
`r2_summary.json` (the paper's Table 4).

### Kaggle setup (do this first)

1. **Create a Kaggle Dataset** containing these files (put them in one folder):
   - `chunks.parquet` (67 MB) - required for BM25
   - `embeddings.npy` (2.7 GB, float32 688992x1024, L2-normalized) - optional, for the hybrid section
   - `queries_50.csv` - required
   - `r2_expansions.json` - required (cached query reformulations)
   - `r2_results.csv` - optional, only for the comparison cell
   - (`rag_v1_milvus.db`, 5.5 GB, only if you set `DENSE_BACKEND='milvus'`; otherwise skip it)
2. **Notebook settings**: Accelerator = GPU (T4 x2 or P100), Internet = **On**
   (needed for `pip install` and Hugging Face model downloads).
3. **Attach the dataset** to this notebook, then run all cells top to bottom.

The notebook auto-detects the attached dataset under `/kaggle/input`; if that fails, set `DATA_DIR`
explicitly in the config cell. Outputs are written to `/kaggle/working/r3_outputs/` and are saved with
the notebook.

**Sections**: (1) BM25 baseline (single vs multi-query RRF fusion), (2) BM25+dense hybrid, (3) dense
replication of Table 4 (precision control), (4) multi-seed expansion stability. Each is switchable in the config cell.


## 1. Install dependencies

Kaggle already ships `torch`, `transformers`, `sentence-transformers`, `pandas`, `pyarrow`, `numpy`,
`scipy`, `accelerate` and `bitsandbytes`. Only `rank_bm25` is missing (and `pymilvus` if you use the
Milvus backend). Requires Internet = On.


In [ ]:
import sys, subprocess

def pip_install(*pkgs, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install", "-q" if quiet else "", *pkgs]
    subprocess.run([c for c in cmd if c], check=True)

pip_install("rank_bm25")
# bitsandbytes >= 0.46.1 enables the 4-bit (nf4) load path for the 7-8B rerankers:
# fp32 needs ~28 GB and fp16 needs ~14.2 GB - neither fits a T4 next to anything else,
# but nf4 fits in ~4.5 GB. Without it the loader falls back to (very slow) CPU fp32.
pip_install("-U", "bitsandbytes>=0.46.1", "accelerate")

# verify - this line MUST appear in the output, and the version must be >= 0.46.1
from importlib.metadata import version as _pkg_version
for _pkg in ("bitsandbytes", "accelerate"):
    try:
        print(f"[installed] {_pkg}=={_pkg_version(_pkg)}")
    except Exception as _e:
        print(f"[MISSING] {_pkg}: {_e}")

# Uncomment ONLY if you use the Milvus dense backend:
# pip_install("pymilvus>=2.4", "milvus-lite")

# If a 7B reranker fails to load because sentence-transformers is too old, uncomment:
# pip_install("-U", "sentence-transformers", "transformers")


## 2. Imports and environment


In [ ]:
import os, re, csv, json, time, pickle, hashlib, glob, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from scipy.spatial.distance import cosine

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
# reduce CUDA allocator fragmentation (name differs across torch versions; both harmless if unknown)
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IS_KAGGLE = Path("/kaggle/input").is_dir()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
print("kaggle:", IS_KAGGLE, "| work dir:", WORK_DIR)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 3. Configuration

Set `DATA_DIR` if auto-detection fails. `MAX_DOCS` / `MAX_QUERIES` are smoke-test switches (0 = full).
`DENSE_BACKEND`: `'embeddings'` (portable, uses `embeddings.npy`), `'milvus'` (needs the DB), or `None`
(skip the hybrid section).


In [ ]:
# ---------------- USER SETTINGS ----------------
DATA_DIR      = None      # e.g. Path("/kaggle/input/rags-c5ae"); None = auto-detect
OUT_DIR       = WORK_DIR / "r3_outputs"
DENSE_BACKEND = "embeddings"   # 'embeddings' | 'milvus' | None

RUN_BM25      = True
RUN_DENSE     = True      # precision control: replicate Table 4's dense/fusion columns in THIS session
RUN_HYBRID    = True      # needs a dense backend
RUN_MULTISEED = False     # needs Mistral-7B; slow on CPU, feasible on GPU
MAX_DOCS      = 0         # 0 = full 688,992 chunks (smoke test: e.g. 50000)
MAX_QUERIES   = 0         # 0 = all 50
N_SEEDS       = 5         # for the multi-seed section

# ---------------- protocol (as in run_r2_benchmark.py) ----------------
TOP_K        = 10
K_RRF        = 60
FUSION_LIMIT = 3          # candidate-list length multiplier before fusion
ENCODER      = "intfloat/multilingual-e5-large-instruct"
SEL          = "multilingual-e5-large"
RERANKER_MODELS = {
    "multilingual-e5-large": "intfloat/multilingual-e5-large-instruct",
    "e5-mistral-7b-instruct": "intfloat/e5-mistral-7b-instruct",
    "SFR-Embedding-Mistral": "Salesforce/SFR-Embedding-Mistral",
    "Linq-Embed-Mistral": "Linq-AI-Research/Linq-Embed-Mistral",
}

# ---------------- BM25 hyperparameters ----------------
BM25_K1 = 1.5
BM25_B  = 0.75

# Prefer 4-bit on the GPU before falling back to (very slow) CPU fp32 for the 7B rerankers.
PREFER_4BIT = IS_KAGGLE

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- resolve data paths ----------------
# Bounded search only: the earlier version ran glob("**/name") over
# cwd/parent/parent, which on Kaggle resolves to "/" (cwd = /kaggle/working)
# and walked the ENTIRE filesystem before returning -> the cell looked hung
# with no output. Never scan the filesystem root; prune hidden/system dirs;
# cap walk depth; print progress as we go.

_SKIP_DIRS = {".git", ".venv", "venv", "__pycache__", "site-packages",
              "node_modules", ".ipynb_checkpoints", ".local", ".cache"}
print("[paths] resolving data paths ...", flush=True)

def _bounded_find(root, name, max_depth=3):
    """Find `name` inside `root` down to max_depth levels (pruned, no symlinks,
    never scanning a filesystem root). Returns Path or None."""
    try:
        root = Path(root).resolve()
    except OSError:
        return None
    if len(root.parts) <= 1:            # "/" itself: never scan
        return None
    direct = root / name
    if direct.is_file():
        return direct
    if not root.is_dir():
        return None
    for dirpath, dirnames, filenames in os.walk(root, followlinks=False):
        rel = Path(dirpath).relative_to(root)
        if len(rel.parts) >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames
                       if d not in _SKIP_DIRS and not d.startswith('.')]
        if name in filenames:
            return Path(dirpath) / name
    return None

_SCAN_LOG = []

def _autodiscover():
    # 1) Kaggle datasets: /kaggle/input/<dataset>/.../chunks.parquet (depth <= 4)
    if IS_KAGGLE and Path("/kaggle/input").is_dir():
        entries = sorted(p for p in Path("/kaggle/input").iterdir())
        print(f"[paths] /kaggle/input has {len(entries)} entries:", flush=True)
        if not entries:
            _SCAN_LOG.append("/kaggle/input is EMPTY -> no dataset attached")
            print("[paths]   (empty -> attach your dataset: Input > Add Input)",
                  flush=True)
        for child in entries:
            hit = _bounded_find(child, "chunks.parquet", max_depth=4)
            msg = f"{child.name}: {'HIT ' + str(hit) if hit else 'no chunks.parquet'}"
            _SCAN_LOG.append(msg)
            print("[paths]   ", msg, flush=True)
            if hit:
                return hit.parent
    # 2) cwd and up two levels (filesystem root skipped by _bounded_find)
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        hit = _bounded_find(base, "chunks.parquet", max_depth=3)
        if hit:
            _SCAN_LOG.append(f"{base}: HIT")
            return hit.parent
        _SCAN_LOG.append(f"{base}: no chunks.parquet")
    return None

if DATA_DIR is not None:
    DATA_DIR = Path(DATA_DIR)
    if not DATA_DIR.is_dir():
        _avail = (sorted(p.name for p in Path("/kaggle/input").iterdir())
                  if Path("/kaggle/input").is_dir() else "(no /kaggle/input)")
        print(f"[paths] DATA_DIR does not exist: {DATA_DIR}", flush=True)
        print(f"[paths] attached inputs: {_avail}", flush=True)
        print("[paths] falling back to auto-detect ...", flush=True)
        DATA_DIR = None

if DATA_DIR is None:
    DATA_DIR = _autodiscover()
assert DATA_DIR is not None, (
    "chunks.parquet not found. What was scanned:\n  - " +
    "\n  - ".join(_SCAN_LOG) +
    "\nFix: attach your Kaggle dataset (Input > Add Input), or set "
    "DATA_DIR = Path('/kaggle/input/<your-dataset>') and re-run.")
DATA_DIR = Path(DATA_DIR)
print("[paths] DATA_DIR  :", DATA_DIR, flush=True)

SEARCH_ROOTS = [DATA_DIR, WORK_DIR]
for _r in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if len(Path(_r).resolve().parts) <= 1:   # never search from "/"
        continue
    if _r not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(_r)
print("[paths] searching :", SEARCH_ROOTS, flush=True)

def _find(name, required=True):
    for root in SEARCH_ROOTS:
        p = Path(root) / name
        if p.exists():
            return p
    for root in SEARCH_ROOTS:
        hit = _bounded_find(root, name, max_depth=3)
        if hit:
            return hit
    if required:
        raise FileNotFoundError(f"{name} not found under {SEARCH_ROOTS}")
    print(f"[paths] optional file not found, skipped: {name}", flush=True)
    return None

CHUNKS_PATH     = _find("chunks.parquet")
QUERIES_PATH    = _find("queries_50.csv")
EXPANSIONS_PATH = _find("r2_expansions.json")
EMBEDDINGS_PATH = _find("embeddings.npy", required=(DENSE_BACKEND == "embeddings"))
MILVUS_DB       = _find("rag_v1_milvus.db", required=False)
R2_RESULTS_PATH = _find("r2_results.csv", required=False)

print("DATA_DIR      :", DATA_DIR, flush=True)
print("chunks        :", CHUNKS_PATH)
print("queries       :", QUERIES_PATH)
print("expansions    :", EXPANSIONS_PATH)
print("embeddings    :", EMBEDDINGS_PATH)
print("r2_results    :", R2_RESULTS_PATH)
print("OUT_DIR       :", OUT_DIR)


## 4. Load corpus, queries, cached expansions


In [ ]:
print("loading corpus ...", flush=True)
texts = pd.read_parquet(CHUNKS_PATH, columns=["text"])["text"].tolist()
if MAX_DOCS:
    texts = texts[:MAX_DOCS]
print("corpus chunks:", len(texts))

queries = pd.read_csv(QUERIES_PATH).to_dict("records")
if MAX_QUERIES:
    queries = queries[:MAX_QUERIES]
print("queries:", len(queries), "|",
      pd.Series([q["lang"] for q in queries]).value_counts().to_dict())

with open(EXPANSIONS_PATH) as f:
    expansions = json.load(f)
missing = [q["id"] for q in queries if q["id"] not in expansions]
assert not missing, f"missing cached expansions for {missing[:5]}..."
print("cached expansion sets:", len(expansions))


## 5. Build the BM25 index

Tokenization is lowercased Unicode word matching (`\w+`), keeping accented French characters and
matching the paper's minimal preprocessing. The index is cached in `OUT_DIR` (keyed by corpus size and
hyperparameters) so re-runs are cheap. At full corpus size the cache is a few hundred MB.


In [ ]:
TOKEN_RE = re.compile(r"\w+", re.UNICODE)
def tokenize(s: str):
    return TOKEN_RE.findall(s.lower())

cache_key = hashlib.md5(f"{len(texts)}|{BM25_K1}|{BM25_B}".encode()).hexdigest()[:10]
cache_path = OUT_DIR / f"bm25_index_{len(texts)}_{cache_key}.pkl"

if cache_path.exists():
    t0 = time.time()
    with open(cache_path, "rb") as f:
        bm25 = pickle.load(f)
    print(f"loaded cached BM25 index in {time.time()-t0:.1f}s -> {cache_path.name}")
else:
    t0 = time.time()
    print(f"building BM25 index over {len(texts):,} chunks "
          f"(first run takes a few minutes; result is cached) ...", flush=True)
    corpus_tokens = []
    for _i, _t in enumerate(texts):
        corpus_tokens.append(tokenize(_t))
        if (_i + 1) % 100_000 == 0:
            print(f"  tokenized {_i+1:,}/{len(texts):,} ({time.time()-t0:.0f}s)",
                  flush=True)
    bm25 = BM25Okapi(corpus_tokens, k1=BM25_K1, b=BM25_B)
    with open(cache_path, "wb") as f:
        pickle.dump(bm25, f)
    print(f"built BM25 index in {time.time()-t0:.1f}s -> {cache_path.name}",
          flush=True)

_s = bm25.get_scores(tokenize("Qui est Antoine Meillet ?"))
for _i in np.argsort(_s)[::-1][:3]:
    print(round(float(_s[_i]), 3), "|", texts[_i][:90])


## 6. Shared helpers (RRF, BM25 ranking, re-scoring, bootstrap CI, model loading)

Chunk `id` = row index in `chunks.parquet` (= Milvus primary key when verified), so sparse and dense
lists fuse on a stable integer id. `rescore`, `bootstrap_ci` and the loading fallback mirror
`run_r2_benchmark.py`; on Kaggle the fallback prefers 4-bit GPU over CPU fp32 for the 7B models.


In [ ]:
def ids_to_texts(ids):
    return [texts[i] for i in ids]

def rrf_fuse_ids(lists, top_k):
    """lists: ranked id lists. Return top_k ids by RRF (k=K_RRF)."""
    scores = {}
    for lst in lists:
        for rank, doc_id in enumerate(lst):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (K_RRF + rank + 1)
    return [i for i, _ in sorted(scores.items(), key=lambda kv: -kv[1])[:top_k]]

def bm25_rank_ids(query, top_n):
    scores = bm25.get_scores(tokenize(query))
    n = min(top_n, len(scores))
    idx = np.argpartition(scores, -n)[-n:]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return [int(i) for i in idx]

def bm25_conditions(query, variations):
    """(single-query BM25, BM25 multi-query RRF fusion) as text lists."""
    single = bm25_rank_ids(query, TOP_K)
    lists = [bm25_rank_ids(q, TOP_K * FUSION_LIMIT) for q in [query] + list(variations)]
    return ids_to_texts(single), ids_to_texts(rrf_fuse_ids(lists, TOP_K))

def rescore(model, query, docs):
    q_vec = model.encode(query, normalize_embeddings=True)
    doc_vecs = model.encode(docs, normalize_embeddings=True)
    scores = [1 - cosine(q_vec, dv) for dv in doc_vecs]
    order = np.argsort(scores)[::-1]
    return {"cos1": float(scores[order[0]]),
            "mean5": float(np.mean([scores[i] for i in order[:5]])),
            "drop": float(scores[order[0]] - scores[order[1]])}

def bootstrap_ci(values, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(n_boot, len(values)))
    means = values[idx].mean(axis=1)
    return [float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]

def _load_4bit(model_cls, path):
    from transformers import BitsAndBytesConfig
    # self-heal: cell 2 may not have run in this kernel/session
    try:
        from importlib.metadata import version as _v
        _bv = tuple(int(x) for x in _v("bitsandbytes").split(".")[:3])
        assert _bv >= (0, 46, 1), f"bitsandbytes {'.'.join(map(str, _bv))} < 0.46.1"
    except Exception as _e:
        print(f"[rerank] bitsandbytes unusable ({_e}) -> installing now ...", flush=True)
        try:
            pip_install("-U", "bitsandbytes>=0.46.1", "accelerate")
            from importlib.metadata import version as _v2
            _nv = _v2("bitsandbytes")
            _bv = tuple(int(x) for x in _nv.split(".")[:3])
            assert _bv >= (0, 46, 1), f"still {_nv} < 0.46.1"
            print(f"[rerank] bitsandbytes now {_nv}", flush=True)
        except Exception as _e2:
            raise RuntimeError(
                f"4-bit unavailable - bitsandbytes install failed: {_e2}") from _e2
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_quant_type="nf4")
    return model_cls(path, device="cuda:0", trust_remote_code=True,
                     model_kwargs={"quantization_config": quant})

def load_reranker(path):
    from sentence_transformers import SentenceTransformer
    def _cuda_fp32():
        return SentenceTransformer(path, device="cuda:0", trust_remote_code=True)
    def _cpu_fp32():
        return SentenceTransformer(path, device="cpu", trust_remote_code=True)
    def _cuda_4bit():
        return _load_4bit(SentenceTransformer, path)
    def _cuda_fp16_auto():
        # fp16 sharded across all visible GPUs via device_map (the T4 x2 path);
        # on a single 14.5 GB GPU a 7B fp16 load OOMs quickly and the chain moves on
        return SentenceTransformer(path, device="cuda", trust_remote_code=True,
                                   model_kwargs={"device_map": "auto",
                                                 "torch_dtype": torch.float16})
    is_large = "mistral" in path.lower()   # e5-mistral / SFR-Mistral / Linq-Mistral = 7-8B
    if is_large:
        # fp32 on GPU is physically impossible for 7-8B (~28 GB) - skip it, no OOM churn
        order = ([_cuda_4bit, _cuda_fp16_auto, _cpu_fp32] if PREFER_4BIT
                 else [_cuda_fp16_auto, _cuda_4bit, _cpu_fp32])
    else:
        order = [_cuda_fp32, _cuda_4bit, _cpu_fp32]
    labels = {_cuda_fp32: "fp32-cuda", _cuda_4bit: "4bit-cuda",
              _cpu_fp32: "fp32-cpu", _cuda_fp16_auto: "fp16-auto"}
    t0 = time.time()
    for attempt in order:
        # release anything left by a failed/partial load BEFORE trying the next path
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        try:
            m = attempt()
            # probe: some paths LOAD but cannot encode (e.g. device_map sharding across
            # 2 GPUs puts layers on cuda:1 while ST feeds cuda:0). Fail HERE, inside
            # the chain, instead of crashing run_condition later.
            m.encode("device probe", normalize_embeddings=True)
            return m, labels[attempt], round(time.time()-t0, 2)
        except Exception as e:
            print(f"[rerank] {labels[attempt]} failed for {path}: {e}", flush=True)
            try:
                del m
            except NameError:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    raise RuntimeError(f"no viable device for {path}")


## 7. Dense backend (for the hybrid section and multi-seed)

`'embeddings'` loads `embeddings.npy` (already L2-normalized, so cosine = dot product) and does exact
top-K search - portable, no Milvus needed (2.7 GB RAM). `'milvus'` queries the collection instead.
Skipped entirely when `RUN_HYBRID` and `RUN_MULTISEED` are both False.


In [ ]:
NEED_DENSE = (RUN_DENSE or RUN_HYBRID or RUN_MULTISEED) and (DENSE_BACKEND is not None)

if NEED_DENSE:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(ENCODER, trust_remote_code=True)

    if DENSE_BACKEND == "embeddings":
        print(f"loading embeddings from {EMBEDDINGS_PATH} (~2.7 GB, a few seconds) ...", flush=True)
        _emb = np.asarray(np.load(str(EMBEDDINGS_PATH)), dtype=np.float32)
        print("dense backend: embeddings", _emb.shape)

        def dense_rank_ids(query, top_n):
            qv = encoder.encode(query, normalize_embeddings=True).astype(np.float32)
            sims = _emb @ qv
            n = min(top_n, sims.shape[0])
            idx = np.argpartition(-sims, n - 1)[:n]
            idx = idx[np.argsort(-sims[idx])]
            return [int(i) for i in idx]

    elif DENSE_BACKEND == "milvus":
        from pymilvus import MilvusClient
        assert MILVUS_DB is not None, "rag_v1_milvus.db not found"
        client = MilvusClient(str(MILVUS_DB))
        assert client.has_collection("rag_v1"), "collection rag_v1 missing"
        client.load_collection("rag_v1")
        print("dense backend: milvus")

        def dense_rank_ids(query, top_n):
            vec = encoder.encode(query, normalize_embeddings=True).tolist()
            hits = client.search(collection_name="rag_v1", data=[vec], anns_field="vector",
                                 search_params={"metric_type": "COSINE"}, limit=top_n,
                                 output_fields=["text"])[0]
            return [int(h["id"]) for h in hits]
    else:
        raise ValueError(DENSE_BACKEND)

    def hybrid_conditions(query, variations):
        L = TOP_K * FUSION_LIMIT
        hybrid = rrf_fuse_ids([dense_rank_ids(query, L), bm25_rank_ids(query, L)], TOP_K)
        qs = [query] + list(variations)
        dense_lists = [dense_rank_ids(q, L) for q in qs]
        bm25_lists = [bm25_rank_ids(q, L) for q in qs]
        return ids_to_texts(hybrid), ids_to_texts(rrf_fuse_ids(dense_lists + bm25_lists, TOP_K))
else:
    print("dense backend not initialized (RUN_HYBRID/RUN_MULTISEED off or DENSE_BACKEND=None)")


## 8. Generic condition runner

Writes a CSV laid out like `r2_results.csv` (`cosine@1_<condA>`, `mean@5_<condA>`, `drop_1to2_<condA>`,
then the same for `<condB>`) and a summary JSON with bootstrap CIs on delta (B minus A).


In [ ]:
def run_condition(condition_fn, out_csv, out_json, condA, condB, label):
    rows = []
    summary = {
        "experiment": label, "n_queries": len(queries), "top_k": TOP_K, "k_rrf": K_RRF,
        "bm25_k1": BM25_K1, "bm25_b": BM25_B, "corpus_size": len(texts),
        "dense_backend": DENSE_BACKEND, "condition_a": condA, "condition_b": condB,
        "models": {}, "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    }
    keys = [f"cosine@1_{condA}", f"mean@5_{condA}", f"drop_1to2_{condA}",
            f"cosine@1_{condB}", f"mean@5_{condB}", f"drop_1to2_{condB}"]
    t_ret = time.time()
    print(f"[r3:{label}] building retrieval lists for {len(queries)} queries "
          f"over {len(texts):,} docs (one-time, before any model loads) ...",
          flush=True)
    retrieved = {q["id"]: condition_fn(q["query"], expansions[q["id"]]) for q in queries}
    print(f"[r3:{label}] retrieval lists ready in {time.time()-t_ret:.1f}s", flush=True)
    for name, path in RERANKER_MODELS.items():
        print(f"[r3:{label}] reranker {name}", flush=True)
        model, precision, load_time = load_reranker(path)
        t0 = time.time()
        per_query = []
        for q in queries:
            a_docs, b_docs = retrieved[q["id"]]
            t_enc = time.time()
            a = rescore(model, q["query"], a_docs)
            b = rescore(model, q["query"], b_docs)
            enc_time = round(time.time() - t_enc, 4)
            per_query.append({
                "qid": q["id"], "category": q["category"], "lang": q["lang"],
                "model": name, "precision": precision,
                keys[0]: round(a["cos1"], 4), keys[1]: round(a["mean5"], 4), keys[2]: round(a["drop"], 4),
                keys[3]: round(b["cos1"], 4), keys[4]: round(b["mean5"], 4), keys[5]: round(b["drop"], 4),
                "rerank_time_s": enc_time,
            })
            if len(per_query) % 10 == 0 or len(per_query) == len(queries):
                print(f"[r3:{label}]   {name}: {len(per_query)}/{len(queries)} "
                      f"queries ({time.time()-t0:.0f}s)", flush=True)
        rows.extend(per_query)
        rerank_total = round(time.time() - t0, 1)
        col = lambda k: np.array([r[k] for r in per_query], dtype=float)
        d1 = col(keys[3]) - col(keys[0])
        d5 = col(keys[4]) - col(keys[1])
        summary["models"][name] = {
            "precision": precision, "load_time_s": load_time,
            "rerank_time_total_s": rerank_total,
            "mean": {k: round(float(col(k).mean()), 4) for k in keys},
            "delta_b_minus_a": {
                "sim@1": round(float(d1.mean()), 4),
                "sim@1_ci95": [round(v, 4) for v in bootstrap_ci(d1)],
                "sim@5": round(float(d5.mean()), 4),
                "sim@5_ci95": [round(v, 4) for v in bootstrap_ci(d5)],
            },
        }
        del model; gc.collect(); torch.cuda.empty_cache()
        print(f"[r3:{label}]   {name}: {precision} load={load_time}s rerank={rerank_total}s", flush=True)
    with open(out_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    with open(out_json, "w") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"[r3:{label}] wrote {out_csv.name} and {out_json.name}")
    return summary


## 9. BM25 baseline

Condition A = single-query BM25; condition B = BM25 over the query and its five cached reformulations,
fused with RRF. This is the W2 sparse-retriever baseline.


In [ ]:
if RUN_BM25:
    bm25_summary = run_condition(
        bm25_conditions,
        OUT_DIR / "bm25_results.csv", OUT_DIR / "bm25_summary.json",
        condA="bm25", condB="bm25fusion", label="bm25",
    )
else:
    print("RUN_BM25=False")


## 10. BM25 + dense hybrid

- `hybrid`    = RRF( dense(q), bm25(q) )
- `hybridexp` = RRF( dense(q)+dense(reformulations), bm25(q)+bm25(reformulations) )


In [ ]:
if RUN_HYBRID and NEED_DENSE:
    hybrid_summary = run_condition(
        hybrid_conditions,
        OUT_DIR / "hybrid_results.csv", OUT_DIR / "hybrid_summary.json",
        condA="hybrid", condB="hybridexp", label="hybrid",
    )
else:
    print("RUN_HYBRID=False or no dense backend")


## 11. Dense replication under this session's precision (control)

Condition A = single-query dense top-10 (Table 4's *dense* column); condition B = multi-query
dense RRF over the cached reformulations (Table 4's *fusion* column). Protocol identical to R2,
but scored with this session's precision chain (fp32 for e5-large, 4-bit for the 7B models,
vs R2's fp16). `dense_summary.json` vs `r2_results.csv` therefore isolates the precision
effect, de-confounding every cross-experiment delta (BM25/hybrid vs R2).


In [ ]:
if RUN_DENSE and NEED_DENSE:
    def dense_conditions(query, variations):
        """A = single-query dense (Table 4 "dense"); B = multi-query dense RRF (Table 4 "fusion)."""
        L = TOP_K * FUSION_LIMIT
        single = dense_rank_ids(query, TOP_K)
        qs = [query] + list(variations)
        fused = rrf_fuse_ids([dense_rank_ids(q, L) for q in qs], TOP_K)
        return ids_to_texts(single), ids_to_texts(fused)

    dense_summary = run_condition(
        dense_conditions,
        OUT_DIR / "dense_results.csv", OUT_DIR / "dense_summary.json",
        condA="dense", condB="fusion", label="dense",
    )
else:
    print("RUN_DENSE=False or no dense backend")


## 12. Compare against the R2 baseline (Table 4)

Headline numbers: does a sparse retriever change the picture, and does adding BM25 to the dense RRF help?


In [ ]:
if R2_RESULTS_PATH is not None:
    r2 = pd.read_csv(R2_RESULTS_PATH)
    r2_sel = r2[r2.model == SEL].mean(numeric_only=True)
    print("=== R2 (dense vs dense+expansion fusion), selected retriever ===")
    print(r2_sel[["cosine@1_dense", "mean@5_dense", "cosine@1_fusion", "mean@5_fusion"]].to_string())
else:
    print("r2_results.csv not provided - skipping the R2 comparison")

for fn, label in [("bm25_summary.json", "BM25"), ("hybrid_summary.json", "HYBRID")]:
    p = OUT_DIR / fn
    if not p.exists():
        print(f"\n[{label}] {fn} not found - run the corresponding section")
        continue
    s = json.load(open(p))
    print(f"\n=== {label}, selected retriever ===")
    print(json.dumps(s["models"][SEL], indent=2, ensure_ascii=False))

# --- precision shift: this session (fp32/4bit) vs R2 (fp16), paired per query ---
if (OUT_DIR / "dense_results.csv").exists() and R2_RESULTS_PATH is not None:
    dnew = pd.read_csv(OUT_DIR / "dense_results.csv")
    r2df = pd.read_csv(R2_RESULTS_PATH)
    print("\n=== precision shift: THIS session minus R2 (same dense/fusion protocol) ===")
    print("    nonzero = fp32/4bit-vs-fp16 effect; read cross-experiment deltas against it")
    for m in RERANKER_MODELS:
        a = dnew[dnew.model == m].set_index("qid")
        b = r2df[r2df.model == m].set_index("qid")
        common = a.index.intersection(b.index)
        for col in ("cosine@1_dense", "mean@5_dense", "cosine@1_fusion", "mean@5_fusion"):
            if col not in a.columns or col not in b.columns or not len(common):
                continue
            d = (a.loc[common, col] - b.loc[common, col]).to_numpy()
            rng = np.random.default_rng(42)
            idx = rng.integers(0, len(d), size=(10000, len(d)))
            ci = np.percentile(d[idx].mean(1), [2.5, 97.5])
            sig = "*" if (ci[0] > 0 or ci[1] < 0) else " "
            print(f"  {m:24s} {col:18s} {d.mean():+.4f}  CI[{ci[0]:+.4f}, {ci[1]:+.4f}] {sig}")
elif not (OUT_DIR / "dense_results.csv").exists():
    print("\n[dense] dense_results.csv not found - run the dense-replication section")


## 13. Multi-seed expansion stability (optional, W3)

The only stochastic component in Table 4 is the expansion LLM (Mistral-7B, temperature 0.7). This
regenerates the reformulations for `N_SEEDS` sampling seeds and re-measures the fusion-minus-dense delta
for the selected retriever, reporting mean +/- sd across seeds (encoders are deterministic).


In [ ]:
if RUN_MULTISEED and NEED_DENSE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    LLM_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

    def load_expansion_llm():
        tok = AutoTokenizer.from_pretrained(LLM_NAME)
        if torch.cuda.is_available():
            quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                       bnb_4bit_quant_type="nf4")
            model = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16,
                                                         device_map="auto",
                                                         quantization_config=quant)
        else:
            model = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float32)
        model.eval()
        return model, tok

    def generate_variations(original_query, model, tok, num_variations=5, max_length=64, seed=0):
        prompt = (f"\n            Reformule la question suivante de {num_variations} manières différentes, "
                  f"\n            en conservant son sens original, dans le but de rechercher des informations dans des archives historiques."
                  f"\n            Chaque reformulation doit être concise et sur une nouvelle ligne. Ne numérote pas les reformulations."
                  f"\n\n            Question originale : {original_query}\n\n            Reformulations :\n        ")
        torch.manual_seed(seed)
        inp = tok(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inp, max_new_tokens=max_length * num_variations, num_return_sequences=1,
                             do_sample=True, temperature=0.7, pad_token_id=tok.pad_token_id)
        text = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        part = text.split("Reformulations :")[-1].strip() if "Reformulations :" in text else text
        return [v.strip() for v in part.split("\n")
                if v.strip() and v.lower() != original_query.lower()][:num_variations]

    model, tok = load_expansion_llm()
    model_rer, precision, _ = load_reranker(RERANKER_MODELS[SEL])
    per_seed = []
    for s in range(N_SEEDS):
        t0 = time.time()
        d1_list, d5_list = [], []
        for q in queries:
            vars_s = generate_variations(q["query"], model, tok, seed=s)
            d_q = dense_rank_ids(q["query"], TOP_K * FUSION_LIMIT)
            lists = [dense_rank_ids(v, TOP_K * FUSION_LIMIT) for v in [q["query"]] + vars_s]
            f_q = ids_to_texts(rrf_fuse_ids(lists, TOP_K))
            a = rescore(model_rer, q["query"], ids_to_texts(d_q))
            b = rescore(model_rer, q["query"], f_q)
            d1_list.append(b["cos1"] - a["cos1"])
            d5_list.append(b["mean5"] - a["mean5"])
        per_seed.append({"seed": s, "delta_sim1": float(np.mean(d1_list)),
                         "delta_sim5": float(np.mean(d5_list))})
        print(f"seed {s}: delta@1={np.mean(d1_list):+.4f} delta@5={np.mean(d5_list):+.4f} "
              f"({time.time()-t0:.0f}s)", flush=True)
    out = {
        "model": SEL, "precision": precision, "n_seeds": N_SEEDS, "per_seed": per_seed,
        "delta_sim1_mean": float(np.mean([p["delta_sim1"] for p in per_seed])),
        "delta_sim1_sd": float(np.std([p["delta_sim1"] for p in per_seed])),
        "delta_sim5_mean": float(np.mean([p["delta_sim5"] for p in per_seed])),
        "delta_sim5_sd": float(np.std([p["delta_sim5"] for p in per_seed])),
    }
    with open(OUT_DIR / "multiseed_expansion.json", "w") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(json.dumps(out, indent=2))
else:
    print("RUN_MULTISEED=False or no dense backend")


## 14. Outputs and results checklist

Everything is written under `/kaggle/working/r3_outputs/` (saved with the notebook as its output).


In [ ]:
print("Files in", OUT_DIR, ":")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:40s} {p.stat().st_size/1e6:8.2f} MB")

print("\nReviewer-point coverage:")
print("  W2 sparse baseline        -> bm25_results.csv / bm25_summary.json")
print("  W2 sparse+dense fusion    -> hybrid_results.csv / hybrid_summary.json")
print("  W3 run-to-run stability   -> multiseed_expansion.json")
